Missing Values and Outliers

Amani Insurance claims case study

Deliverable: To identify and handle missing values and outliers.

In [2]:
import numpy as np
import pandas as pd

FILE_PATH = "insurance_claims_messy.csv"

IDENTIFY MISSING VALUES

In [3]:
df = pd.read_csv(FILE_PATH)

# Same comma-strip + coercion as Part 1 -- blanks and stray garbage both
# become NaN (pandas' missing-value marker) via errors="coerce".
df["claim_amount_kes"] = (
    df["claim_amount_kes"].astype(str).str.replace(",", "", regex=False).str.strip()
)
df["claim_amount_kes"] = pd.to_numeric(df["claim_amount_kes"], errors="coerce")

# .isna() returns True/False per cell; .sum() on that counts the Trues --
# i.e. this gives us a missing-value count per column in one line.
df.isna().sum()

claim_id              0
policy_number         0
claim_type            0
claim_amount_kes     30
claim_date            0
region                0
status                0
assessor              0
notes               190
dtype: int64

In [5]:
pct_missing = df["claim_amount_kes"].isna().mean() * 100
print(f"Percentage of rows missing claim_amount_kes: {pct_missing:.1f}%")

Percentage of rows missing claim_amount_kes: 5.9%


Handling missing values

In [7]:
df["claim_type_clean"] = (
    df["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
)

# (a) Drop every row with a missing amount.
dropped = df.dropna(subset=["claim_amount_kes"])
print(f"(a) Drop: {len(df) - len(dropped)} rows removed, {len(dropped)} remain.")

# (b) Fill every missing value with the SAME overall median.
overall_median = df["claim_amount_kes"].median()
filled_overall = df["claim_amount_kes"].fillna(overall_median)
print(f"(b) Fill with overall median (KES {overall_median:,.0f}).")

# (c) Fill each row's missing value with THAT ROW'S claim-type median.
# groupby(...).transform("median") returns one value per ROW (not per group),
# so it lines up directly with df's own index and can be passed to fillna().
group_median = df.groupby("claim_type_clean")["claim_amount_kes"].transform("median")
filled_by_group = df["claim_amount_kes"].fillna(group_median)
print("(c) Fill with per-claim-type median -- e.g. a missing health-outpatient")
print("    claim gets filled with the health-outpatient median, not the")
print("    whole-dataset median.")

(a) Drop: 30 rows removed, 479 remain.
(b) Fill with overall median (KES 200,500).
(c) Fill with per-claim-type median -- e.g. a missing health-outpatient
    claim gets filled with the health-outpatient median, not the
    whole-dataset median.


Identifying outliers with IQR method

In [8]:
def iqr_bounds(series, k=1.5):
    """Return (lower, upper) outlier bounds using Q1 - k*IQR / Q3 + k*IQR.
    k=1.5 is the standard boxplot default."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

In [9]:
# Start with every row marked "not an outlier" (False), then flip specific
# rows to True as we find them, one claim type at a time.
outlier_flags = pd.Series(False, index=df.index)

for claim_type, group in df.groupby("claim_type_clean"):
    valid = group["claim_amount_kes"].dropna()
    if len(valid) < 5:      # too few data points to trust a quartile calculation
        continue
    lower, upper = iqr_bounds(valid)
    is_outlier = (group["claim_amount_kes"] < lower) | (group["claim_amount_kes"] > upper)
    outlier_flags.loc[group.index] = is_outlier.fillna(False)
    n_out = is_outlier.sum()
    if n_out:
        print(f"{claim_type:20s} bounds=({lower:>10,.0f}, {upper:>12,.0f})  outliers found: {n_out}")

df["is_outlier"] = outlier_flags
print(f"\nTotal rows flagged as outliers: {df['is_outlier'].sum()} of {len(df)}")

health-inpatient     bounds=(   -59,200,      275,400)  outliers found: 2
health-outpatient    bounds=(    -3,750,       19,850)  outliers found: 1
motor-accident       bounds=(     8,012,      349,512)  outliers found: 4
motor-theft          bounds=(   147,400,    1,190,600)  outliers found: 3
property-burglary    bounds=(  -162,738,      673,162)  outliers found: 1
property-fire        bounds=(   -97,200,    1,853,200)  outliers found: 4

Total rows flagged as outliers: 15 of 509


Handling outliers

In [10]:
# A claim amount can NEVER legitimately be negative -- this is a sanity
# rule specific to the domain, not a statistical test, and it catches
# errors that IQR alone might miss.
negative_mask = df["claim_amount_kes"] < 0
print(f"Rows with a NEGATIVE claim_amount_kes (unambiguous errors): {negative_mask.sum()}")
df.loc[negative_mask, ["claim_id", "claim_type_clean", "claim_amount_kes"]].head()

Rows with a NEGATIVE claim_amount_kes (unambiguous errors): 7


,claim_id,claim_type_clean,claim_amount_kes
31,CLM-00109,motor-theft,-740500.0
52,CLM-00352,motor-theft,-1101100.0
165,CLM-00269,property-burglary,-355100.0
198,CLM-00376,property-fire,-1325300.0
247,CLM-00137,health-outpatient,-8900.0


In [11]:
# Compare: these are flagged by STATISTICS, but aren't necessarily wrong --
# each one needs a human look, not an automatic delete.
df.loc[df["is_outlier"] & ~negative_mask,
       ["claim_id", "claim_type_clean", "claim_amount_kes"]].head()

,claim_id,claim_type_clean,claim_amount_kes
49,CLM-00422,motor-accident,837000.0
91,CLM-00249,property-fire,8307000.0
183,CLM-00026,motor-accident,500.0
283,CLM-00328,motor-accident,1293000.0
363,CLM-00327,motor-accident,500.0


In [12]:
def clean_claims(filepath):
    """Load and clean the Amani claims CSV, returning a DataFrame ready
    for analysis. Fixes applied:
        - claim_amount_kes: comma-stripped, coerced to numeric
        - negative amounts: treated as sign-entry errors -> made positive
        - missing amounts: filled with that claim type's median
        - claim_type: canonicalised to lower-case, hyphenated form
        - status/region: stripped + canonicalised casing
        - an `is_outlier` flag (IQR method, per claim type) is kept as a
          column rather than used to silently drop rows -- downstream
          analysis can decide whether to exclude flagged rows or not.
    """
    data = pd.read_csv(filepath)

    data["claim_amount_kes"] = (
        data["claim_amount_kes"].astype(str).str.replace(",", "", regex=False).str.strip()
    )
    data["claim_amount_kes"] = pd.to_numeric(data["claim_amount_kes"], errors="coerce")

    # Negative amounts are a sign-entry error in this domain -- flip them positive.
    data["claim_amount_kes"] = data["claim_amount_kes"].abs()

    # Canonicalise the categorical columns.
    data["claim_type"] = (
        data["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
    )
    data["status"] = data["status"].astype(str).str.strip().str.lower().str.replace("-", " ")
    data["region"] = data["region"].astype(str).str.strip().str.title()

    # Fill missing amounts with the PER-CLAIM-TYPE median (Section 2, strategy c).
    group_median = data.groupby("claim_type")["claim_amount_kes"].transform("median")
    data["claim_amount_kes"] = data["claim_amount_kes"].fillna(group_median)

    # Drop exact duplicate rows (the same claim logged twice by mistake).
    before = len(data)
    data = data.drop_duplicates()
    print(f"dropped {before - len(data)} exact duplicate rows")

    # Flag outliers per claim type -- kept as a column, never used to delete rows.
    flags = pd.Series(False, index=data.index)
    for claim_type, group in data.groupby("claim_type"):
        lower, upper = iqr_bounds(group["claim_amount_kes"])
        is_out = (group["claim_amount_kes"] < lower) | (group["claim_amount_kes"] > upper)
        flags.loc[group.index] = is_out
    data["is_outlier"] = flags

    return data

In [13]:
cleaned = clean_claims(FILE_PATH)

print(f"\nFinal cleaned shape: {cleaned.shape}")
print(f"Remaining missing claim_amount_kes: {cleaned['claim_amount_kes'].isna().sum()}")
print(f"Remaining negative claim_amount_kes: {(cleaned['claim_amount_kes'] < 0).sum()}")
print(f"Outliers flagged (kept, not dropped): {cleaned['is_outlier'].sum()}")

dropped 9 exact duplicate rows

Final cleaned shape: (500, 10)
Remaining missing claim_amount_kes: 0
Remaining negative claim_amount_kes: 0
Outliers flagged (kept, not dropped): 15
